In [ ]:
# ===========================
# 完整本地 MLflow + 决策树模板
# ===========================

# 1️⃣ 安装必要库（第一次运行时）
!pip install pandas scikit-learn mlflow

# 2️⃣ 导入库
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import mlflow
import mlflow.sklearn

# 3️⃣ 读取本地 CSV（确保文件在 Notebook 同目录）
df = pd.read_csv("employee_attrition.csv")  # 修改成你的文件名
df.head()

# 4️⃣ 数据准备
# 假设 'Attrition' 是目标列，其他都是特征
X = df.drop(columns=["Attrition"])
y = df["Attrition"].apply(lambda x: 1 if x == "Yes" else 0)  # 转成 0/1

# 5️⃣ 划分训练集/测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 6️⃣ 设置 MLflow 实验
mlflow.set_experiment("DecisionTree_EmployeeAttrition")

# 7️⃣ 定义一个训练函数，便于多次实验
def train_decision_tree(max_depth, min_samples_split):
    with mlflow.start_run():
        # 初始化模型
        dt = DecisionTreeClassifier(max_depth=max_depth, min_samples_split=min_samples_split, random_state=42)
        dt.fit(X_train, y_train)
        
        # 预测并计算准确率
        y_pred = dt.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        
        # 记录参数和指标
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("min_samples_split", min_samples_split)
        mlflow.log_metric("accuracy", acc)
        
        # 注册模型
        mlflow.sklearn.log_model(
            sk_model=dt,
            artifact_path="decision_tree_model",
            registered_model_name="DecisionTree_EmployeeAttrition_Model"
        )
        
        print(f"Trained DecisionTree(max_depth={max_depth}, min_samples_split={min_samples_split}) -> accuracy={acc:.4f}")

# 8️⃣ 训练实验 1
train_decision_tree(max_depth=4, min_samples_split=2)

# 9️⃣ 训练实验 2（微调参数）
train_decision_tree(max_depth=6, min_samples_split=4)

# 10️⃣ 启动本地 MLflow UI 查看结果
# 在终端/命令行运行，不用在 Notebook 里：
# mlflow ui
# 然后打开浏览器访问 http://127.0.0.1:5000